# Figuras do artigo — DeepPCB

Notebook auxiliar dedicado exclusivamente à construção das figuras do artigo. Ele não treina modelos, não escolhe estratégias e não executa inferência.

- **Figura 1:** imagem inspecionada, template e anotação de referência.
- **Figura 2:** curvas de mAP de validação e perda de treinamento das estratégias A, B e C.
- **Figura 3:** dois exemplos qualitativos previamente produzidos pela avaliação final.

Quando um ID ou diretório estiver configurado como `None`, o notebook utiliza deterministicamente a primeira opção válida, sem levantar ou ranquear candidatos.

## 1. Configuração

In [ ]:
CONFIG = {
    'drive_project_dir': '/content/drive/MyDrive/vision-transfer-learning',
    'colab_dataset_dir': 'dataset/DeepPCB/PCBData',
    'local_dataset_dir': 'data/DeepPCB/PCBData',
    'results_dir': 'runs/results',
    'comparison_manifest': 'runs/results/comparison/comparison_manifest.json',
    'output_dir': 'article/figures',

    # Figura 1: None usa a primeira imagem de trainval.txt.
    'fig01_image_id': None,

    # Figura 3: None usa a primeira execução que contenha classified_imgs_index.csv.
    'fig03_test_run_dir': None,
    # None usa a primeira imagem do índice. O segundo ID usa a primeira imagem diferente.
    'fig03_first_image_id': None,
    'fig03_second_image_id': None,
}

STRATEGY_COLORS = {
    'A': '#4C78A8',
    'B': '#F58518',
    'C': '#54A24B',
}

## 2. Ambiente e caminhos

In [ ]:
import json
from pathlib import Path, PurePosixPath

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from PIL import Image

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount('/content/drive')

if IN_COLAB:
    PROJECT_DIR = Path(CONFIG['drive_project_dir']).expanduser().resolve()
    DATASET_DIR = (PROJECT_DIR / CONFIG['colab_dataset_dir']).resolve()
else:
    roots = [Path.cwd(), *Path.cwd().parents]
    PROJECT_DIR = next((root for root in roots if (root / 'pyproject.toml').is_file()), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError('Raiz do projeto não encontrada. Execute a partir de vision-transfer-learning.')
    DATASET_DIR = (PROJECT_DIR / CONFIG['local_dataset_dir']).resolve()

RESULTS_DIR = (PROJECT_DIR / CONFIG['results_dir']).resolve()
OUTPUT_DIR = (PROJECT_DIR / CONFIG['output_dir']).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f'Diretório DeepPCB não encontrado: {DATASET_DIR}')
if not RESULTS_DIR.is_dir():
    raise FileNotFoundError(f'Diretório de resultados não encontrado: {RESULTS_DIR}')

def project_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (PROJECT_DIR / path).resolve()

print('Ambiente:', 'Google Colab' if IN_COLAB else 'local')
print('Projeto:', PROJECT_DIR)
print('Dataset:', DATASET_DIR)
print('Resultados:', RESULTS_DIR)
print('Figuras:', OUTPUT_DIR)

## 3. Figura 1 — Exemplo do conjunto DeepPCB

A figura usa somente `trainval.txt`. Se `fig01_image_id` estiver vazio, é utilizada a primeira entrada do arquivo.

In [ ]:
CLASSES = {
    1: 'open',
    2: 'short',
    3: 'mousebite',
    4: 'spur',
    5: 'copper',
    6: 'pin-hole',
}


def read_trainval(dataset_dir):
    images, annotations = [], []
    split_path = dataset_dir / 'trainval.txt'
    if not split_path.is_file():
        raise FileNotFoundError(f'trainval.txt não encontrado: {split_path}')

    for line_number, line in enumerate(split_path.read_text(encoding='utf-8').splitlines(), start=1):
        parts = line.split()
        if len(parts) != 2:
            raise ValueError(f'Linha inválida em trainval.txt ({line_number}): {line!r}')
        image_ref, annotation_ref = (PurePosixPath(part) for part in parts)
        base = dataset_dir.joinpath(*image_ref.parts).with_suffix('')
        annotation_path = dataset_dir.joinpath(*annotation_ref.parts)
        image_path = Path(f'{base}_test.jpg')
        template_path = Path(f'{base}_temp.jpg')
        missing = [path for path in (image_path, template_path, annotation_path) if not path.is_file()]
        if missing:
            raise FileNotFoundError(f'Arquivos ausentes para {image_ref}: {missing}')

        image_id = image_ref.stem
        images.append({
            'image_id': image_id,
            'image_path': image_path,
            'template_path': template_path,
            'annotation_path': annotation_path,
        })

        for annotation_line, raw in enumerate(annotation_path.read_text(encoding='utf-8').splitlines(), start=1):
            values = raw.split()
            if len(values) != 5:
                raise ValueError(f'Anotação inválida em {annotation_path}, linha {annotation_line}: {raw!r}')
            x1, y1, x2, y2, label = (float(value) for value in values)
            label = int(label)
            if label not in CLASSES:
                raise ValueError(f'Classe desconhecida {label} em {annotation_path}')
            annotations.append({
                'image_id': image_id,
                'class_id': label,
                'class_name': CLASSES[label],
                'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
            })

    return pd.DataFrame(images), pd.DataFrame(annotations)


IMAGES, ANNOTATIONS = read_trainval(DATASET_DIR)
SELECTED_FIG01_ID = CONFIG['fig01_image_id']
if SELECTED_FIG01_ID is None:
    SELECTED_FIG01_ID = str(IMAGES.iloc[0].image_id)
    print(f'Figura 1: usando a primeira imagem de trainval: {SELECTED_FIG01_ID}')
if SELECTED_FIG01_ID not in set(IMAGES.image_id):
    raise ValueError(f'Imagem não encontrada em trainval: {SELECTED_FIG01_ID}')

fig01_row = IMAGES.loc[IMAGES.image_id == SELECTED_FIG01_ID].iloc[0]
fig01_boxes = ANNOTATIONS.loc[ANNOTATIONS.image_id == SELECTED_FIG01_ID].copy()
print('Imagem:', fig01_row.image_path)
print('Template:', fig01_row.template_path)
print('Anotação:', fig01_row.annotation_path)

In [ ]:
def add_reference_boxes(axis, boxes):
    for box in boxes.itertuples():
        color = plt.cm.tab10(box.class_id - 1)
        axis.add_patch(Rectangle(
            (box.x1, box.y1), box.x2 - box.x1, box.y2 - box.y1,
            fill=False, edgecolor=color, linewidth=1.4,
        ))
        axis.text(
            box.x1, max(0, box.y1 - 4), box.class_name,
            color='white', fontsize=5.5,
            bbox={'facecolor': color, 'edgecolor': 'none', 'alpha': 0.9, 'pad': 1},
        )

with Image.open(fig01_row.image_path) as image:
    inspected_image = np.array(image.convert('RGB'))
with Image.open(fig01_row.template_path) as image:
    template_image = np.array(image.convert('RGB'))

fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.55))
axes[0].imshow(inspected_image)
axes[0].set_title('(a) Imagem inspecionada', fontsize=8)
axes[1].imshow(template_image)
axes[1].set_title('(b) Template de referência', fontsize=8)
axes[2].imshow(inspected_image)
add_reference_boxes(axes[2], fig01_boxes)
axes[2].set_title('(c) Anotação de referência', fontsize=8)
for axis in axes:
    axis.axis('off')
fig.subplots_adjust(left=0.005, right=0.995, top=0.91, bottom=0.01, wspace=0.025)
fig01_pdf = OUTPUT_DIR / 'fig01_dataset_example.pdf'
fig01_png = OUTPUT_DIR / 'fig01_dataset_example.png'
fig.savefig(fig01_pdf, bbox_inches='tight', pad_inches=0.02)
fig.savefig(fig01_png, dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()

fig01_manifest = {
    'image_id': SELECTED_FIG01_ID,
    'image_path': str(fig01_row.image_path),
    'template_path': str(fig01_row.template_path),
    'annotation_path': str(fig01_row.annotation_path),
    'defect_count': int(len(fig01_boxes)),
    'classes': {
        str(name): int(count)
        for name, count in fig01_boxes.class_name.value_counts().sort_index().items()
    },
}
(OUTPUT_DIR / 'fig01_dataset_example.json').write_text(
    json.dumps(fig01_manifest, indent=2, ensure_ascii=False), encoding='utf-8'
)
print('Figura 1:', fig01_pdf)

## 4. Figura 2 — Curvas de convergência

As três rodadas são recuperadas do manifesto produzido pelo notebook de comparação.

In [ ]:
comparison_manifest_path = project_path(CONFIG['comparison_manifest'])
if not comparison_manifest_path.is_file():
    raise FileNotFoundError(f'Manifesto da comparação não encontrado: {comparison_manifest_path}')
comparison_manifest = json.loads(comparison_manifest_path.read_text(encoding='utf-8'))

def history_path_from_manifest(strategy):
    run = comparison_manifest['runs'][strategy]
    recorded = Path(run['files']['history']['path'])
    if recorded.is_file():
        return recorded
    fallback = RESULTS_DIR / Path(run['directory']).name / 'history.csv'
    if fallback.is_file():
        return fallback
    raise FileNotFoundError(f'Histórico da estratégia {strategy} não encontrado.')

HISTORIES = {}
HISTORY_PATHS = {}
for strategy in ('A', 'B', 'C'):
    path = history_path_from_manifest(strategy)
    history = pd.read_csv(path).sort_values('epoch').reset_index(drop=True)
    required = {'epoch', 'train_loss', 'map_50_95'}
    missing = required - set(history.columns)
    if missing:
        raise ValueError(f'Histórico {strategy} sem colunas: {sorted(missing)}')
    HISTORY_PATHS[strategy] = path
    HISTORIES[strategy] = history
    print(f'{strategy}: {path}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.16, 2.8))
for strategy in ('A', 'B', 'C'):
    history = HISTORIES[strategy]
    color = STRATEGY_COLORS[strategy]
    axes[0].plot(
        history['epoch'], history['map_50_95'],
        marker='o', markersize=2.8, linewidth=1.2,
        color=color, label=f'Estratégia {strategy}',
    )
    best = history.loc[history['map_50_95'].idxmax()]
    axes[0].scatter(
        best['epoch'], best['map_50_95'],
        s=30, color=color, edgecolor='black', linewidth=0.6, zorder=3,
    )
    axes[1].plot(
        history['epoch'], history['train_loss'],
        marker='o', markersize=2.8, linewidth=1.2,
        color=color, label=f'Estratégia {strategy}',
    )

axes[0].set_title('(a) mAP de validação', fontsize=8.5)
axes[0].set_xlabel('Época', fontsize=7.5)
axes[0].set_ylabel('mAP@0.50:0.95', fontsize=7.5)
axes[0].set_ylim(0, 1.02)
axes[1].set_title('(b) Perda de treinamento', fontsize=8.5)
axes[1].set_xlabel('Época', fontsize=7.5)
axes[1].set_ylabel('Perda média', fontsize=7.5)
for axis in axes:
    axis.grid(alpha=0.3, linewidth=0.5)
    axis.tick_params(labelsize=7)
    axis.legend(fontsize=6.5, frameon=True)
fig.tight_layout(w_pad=1.4)
fig02_pdf = OUTPUT_DIR / 'fig02_training_curves.pdf'
fig02_png = OUTPUT_DIR / 'fig02_training_curves.png'
fig.savefig(fig02_pdf, bbox_inches='tight', pad_inches=0.03)
fig.savefig(fig02_png, dpi=300, bbox_inches='tight', pad_inches=0.03)
plt.show()

fig02_manifest = {
    'comparison_manifest': str(comparison_manifest_path),
    'histories': {strategy: str(path) for strategy, path in HISTORY_PATHS.items()},
    'best_epochs': {
        strategy: int(history.loc[history['map_50_95'].idxmax(), 'epoch'])
        for strategy, history in HISTORIES.items()
    },
}
(OUTPUT_DIR / 'fig02_training_curves.json').write_text(
    json.dumps(fig02_manifest, indent=2, ensure_ascii=False), encoding='utf-8'
)
print('Figura 2:', fig02_pdf)

## 5. Figura 3 — Predições qualitativas

A figura combina duas imagens já produzidas pelo teste final. Os IDs devem ser fixados em `CONFIG`; quando ausentes, são usadas as duas primeiras entradas distintas do índice.

In [ ]:
def resolve_test_run():
    configured = CONFIG['fig03_test_run_dir']
    if configured is not None:
        path = project_path(configured)
        if not path.is_dir():
            raise FileNotFoundError(f'Rodada de teste não encontrada: {path}')
        return path
    indexes = sorted(
        path for path in RESULTS_DIR.rglob('classified_imgs_index.csv')
        if path.parent.name == 'classified_imgs'
    )
    if not indexes:
        raise FileNotFoundError('Nenhum classified_imgs_index.csv foi encontrado em runs/results.')
    selected = indexes[0].parent.parent
    print(f'Figura 3: usando a primeira rodada encontrada: {selected}')
    return selected

FIG03_RUN_DIR = resolve_test_run()
FIG03_IMAGES_DIR = FIG03_RUN_DIR / 'classified_imgs'
FIG03_INDEX_PATH = FIG03_IMAGES_DIR / 'classified_imgs_index.csv'
if not FIG03_INDEX_PATH.is_file():
    raise FileNotFoundError(f'Índice das imagens classificadas não encontrado: {FIG03_INDEX_PATH}')
FIG03_INDEX = pd.read_csv(FIG03_INDEX_PATH, dtype={'image_id': str, 'file': str})
if FIG03_INDEX.empty:
    raise ValueError('classified_imgs_index.csv está vazio.')

FIRST_FIG03_ID = CONFIG['fig03_first_image_id']
if FIRST_FIG03_ID is None:
    FIRST_FIG03_ID = str(FIG03_INDEX.iloc[0].image_id)
    print(f'Figura 3, primeiro exemplo: usando a primeira imagem: {FIRST_FIG03_ID}')

SECOND_FIG03_ID = CONFIG['fig03_second_image_id']
if SECOND_FIG03_ID is None:
    remaining = FIG03_INDEX.loc[FIG03_INDEX.image_id != FIRST_FIG03_ID]
    SECOND_FIG03_ID = str((remaining.iloc[0] if not remaining.empty else FIG03_INDEX.iloc[0]).image_id)
    print(f'Figura 3, segundo exemplo: usando a primeira imagem diferente: {SECOND_FIG03_ID}')

def classified_image_path(image_id):
    rows = FIG03_INDEX.loc[FIG03_INDEX.image_id == str(image_id)]
    if rows.empty:
        raise ValueError(f'ID não encontrado no índice de imagens classificadas: {image_id}')
    path = FIG03_IMAGES_DIR / rows.iloc[0].file
    if not path.is_file():
        raise FileNotFoundError(f'Imagem classificada não encontrada: {path}')
    return path

FIG03_FIRST_PATH = classified_image_path(FIRST_FIG03_ID)
FIG03_SECOND_PATH = classified_image_path(SECOND_FIG03_ID)
print('Primeiro exemplo:', FIG03_FIRST_PATH)
print('Segundo exemplo:', FIG03_SECOND_PATH)

In [ ]:
with Image.open(FIG03_FIRST_PATH) as image:
    first_example = np.array(image.convert('RGB'))
with Image.open(FIG03_SECOND_PATH) as image:
    second_example = np.array(image.convert('RGB'))

fig, axes = plt.subplots(2, 1, figsize=(7.16, 7.5))
axes[0].imshow(first_example)
axes[0].set_title(f'(a) Primeiro exemplo — {FIRST_FIG03_ID}', fontsize=8.5)
axes[1].imshow(second_example)
axes[1].set_title(f'(b) Segundo exemplo — {SECOND_FIG03_ID}', fontsize=8.5)
for axis in axes:
    axis.axis('off')
fig.subplots_adjust(left=0.005, right=0.995, top=0.97, bottom=0.005, hspace=0.08)
fig03_pdf = OUTPUT_DIR / 'fig03_qualitative_predictions.pdf'
fig03_png = OUTPUT_DIR / 'fig03_qualitative_predictions.png'
fig.savefig(fig03_pdf, bbox_inches='tight', pad_inches=0.03)
fig.savefig(fig03_png, dpi=300, bbox_inches='tight', pad_inches=0.03)
plt.show()

fig03_manifest = {
    'test_run_dir': str(FIG03_RUN_DIR),
    'index_path': str(FIG03_INDEX_PATH),
    'first_image_id': FIRST_FIG03_ID,
    'first_image_path': str(FIG03_FIRST_PATH),
    'second_image_id': SECOND_FIG03_ID,
    'second_image_path': str(FIG03_SECOND_PATH),
}
(OUTPUT_DIR / 'fig03_qualitative_predictions.json').write_text(
    json.dumps(fig03_manifest, indent=2, ensure_ascii=False), encoding='utf-8'
)
print('Figura 3:', fig03_pdf)

## 6. Arquivos produzidos

Após a execução completa, `article/figures/` contém as Figuras 1–3 em PDF e PNG, além de um manifesto JSON para cada figura. Para a versão definitiva, fixe todos os IDs e o diretório da rodada de teste em `CONFIG`.

In [ ]:
for path in sorted(OUTPUT_DIR.glob('fig0*')):
    print(path.name)